# Q-SENTINEL D5 — Threshold Calibration
This notebook reproduces the statistical calibration and 100,000-trial FAR validation used by the D1–D6 acceptance report.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "qsentinel").exists():
    ROOT = next(p for p in Path.cwd().parents if (p / "qsentinel").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qsentinel.statistics import calibrate, validate_false_alarm_rate

## Closed-form threshold
For a Bernoulli mismatch baseline $p$ and target one-sided error budget $\alpha$, the deployed threshold uses the Hoeffding expression
$p_{th}=p+\sqrt{\ln(1/\alpha)/(2n)}$.

In [ ]:
cal = calibrate(
    128,
    baseline_mismatch=0.05,
    target_far=0.01,
    target_frr=0.01,
)
cal

## 100,000-trial empirical FAR validation
The exact binomial tail is the finite-sample operating-point reference. The Hoeffding number is retained as a conservative provable upper bound, so the two quantities are intentionally not treated as equal.

In [ ]:
report = validate_false_alarm_rate(
    n=128,
    baseline_rate=0.05,
    threshold_count=13,
    trials=100_000,
    seed=2026,
)
print(report)
print("Empirical FAR within 10% of exact analytic FAR:", report.within_ten_percent)
print("Empirical FAR below Hoeffding bound:", report.empirical_far <= report.hoeffding_bound)

## Expected acceptance result
With the fixed seed used in CI, the empirical FAR is close to the exact binomial tail and the result is reproducible.